### Install and Setup

In [36]:
pip install google-genai pandas sklearn jsonschema tqdm

  Using cached jsonschema-4.25.1-py3-none-any.whl (90 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
  Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
  Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using legacy 'setup.py install' for sklearn, since package 'wheel' is not installed.
    Running setup.py install for sklearn: started
    Running setup.py install for sklearn: finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


    ERROR: Command errored out with exit status 1:
     command: 'c:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe' -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'C:\\Users\\HP\\AppData\\Local\\Temp\\pip-install-_a4o4ewr\\sklearn_379ed89a5f1b4d0ca0218879d47d2490\\setup.py'"'"'; __file__='"'"'C:\\Users\\HP\\AppData\\Local\\Temp\\pip-install-_a4o4ewr\\sklearn_379ed89a5f1b4d0ca0218879d47d2490\\setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base 'C:\Users\HP\AppData\Local\Temp\pip-pip-egg-info-rl3jfve7'
         cwd: C:\Users\HP\AppData\Local\Temp\pip-install-_a4o4ewr\sklearn_379ed89a5f1b4d0ca0218879d47d2490\
    Complete output (15 lines):
    The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
    rather than '

In [63]:

pip install openai

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


###  Import Libraries

In [40]:
import pandas as pd
import json
from google import genai
from tqdm import tqdm
from jsonschema import validate, ValidationError
from sklearn.metrics import accuracy_score


### API Setup (Gemini)

In [ ]:
client = genai.Client(api_key="GENAI_API_KEY") 
MODEL_NAME = "gemini-2.5-flash"


In [ ]:
# from openai import OpenAI
# import json
# import re

# client = OpenAI(
#     api_key="OPENAI_API_KEY"
# )

# MODEL_NAME = "gpt-5-nano"  



### LLM call Function

In [74]:
def call_llm(prompt):
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt + "\n\nReturn ONLY valid JSON. No explanation. No extra text."
        )
        return response.text  
    except Exception as e:
        print("LLM Error:", str(e))
        return None


In [ ]:
# for openai api (limit reached for free versions)
def call_llm(prompt):
    try:
        full_prompt = prompt + "\n\nReturn ONLY valid JSON. No explanation. No extra text."
        
        response = client.responses.create(
            model=MODEL_NAME,
            input=full_prompt,
            store=True
        )
        
        output_text = response.output_text
        return output_text.strip()
    
    except Exception as e:
        print("LLM Error:", str(e))
        return None


### JSON Extractor 

In [75]:
import re

def extract_json(text):
    if text is None:
        return None
    
    codeblock = re.search(r"```json(.*?)```", text, re.DOTALL)
    if codeblock:
        text = codeblock.group(1).strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except:
            return None
    
    try:
        return json.loads(text)
    except:
        return None


### Load Dataset and sample it (200 rows)

In [76]:
df = pd.read_csv("data/yelp.csv")  

sample = df.sample(15, random_state=42).reset_index(drop=True)

sample.head()


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,QVR7dsvBeg8xFt9B-vd1BA,22-07-2010,hwYVJs8Ko4PMjI19QcR57g,4,We got here around midnight last Friday... the...,review,90a6z--_CUrl84aCzZyPsg,5,5,2
1,24qSrF_XOrvaHDBy-gLIQg,22-01-2012,0mvthYPKb2ZmKhCADiKSmQ,5,Brought a friend from Louisiana here. She say...,review,9lJAj_2zCvP2jcEiRjF9oA,0,0,0
2,j0Uc-GuOe-x9_N_IK1KPpA,09-05-2009,XJHknNIecha6h0wkBSZB4w,3,"Every friday, my dad and I eat here. We order ...",review,0VfJi9Au0rVFVnPKcJpt3Q,0,0,0
3,RBiiGw8c7j-0a8nk35JO3w,22-12-2010,z6y3GRpYDqTznVe-0dn--Q,1,"My husband and I were really, really disappoin...",review,lwppVF0Yqkuwt-xaEuugqw,2,2,2
4,U8VA-RW6LYOhxR-Ygi6eDw,17-01-2011,vhWHdemMvsqVNv5zi2OMiA,5,Love this place! Was in phoenix 3 weeks for w...,review,Y2R_tlSk4lTHiLXTDsn1rg,0,1,0


### Prompt Version 1 — Basic Prompt
Simple direct instruction.  
Goal: evaluate baseline accuracy & JSON validity.


In [77]:
def prompt_v1(review):
    return f"""

Review: "{review}"

Return EXACT format:
{{
  "predicted_stars": <number between 1 and 5>,
  "explanation": "short explanation"
}}
"""


Running the Prompt V1 for data sample (200)

In [88]:
results_v1 = []

for i, row in tqdm(sample.iterrows(), total=len(sample)):
    full_prompt = prompt_v1(row["text"])
    raw = call_llm(full_prompt)

    try:
        parsed = extract_json(raw)
        predicted = parsed.get("predicted_stars", None)
        explanation = parsed.get("explanation", None)
        valid_json = True
    except:
        predicted = None
        valid_json = False
    
    results_v1.append({
        "actual": row["stars"],
        "pred": predicted,
        "json_valid": valid_json,
        "explanation":explanation,
        "actual text": row["text"]
    })

df_v1 = pd.DataFrame(results_v1)
df_v1


100%|██████████| 15/15 [00:46<00:00,  3.12s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 41.205853849s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2

,actual,pred,json_valid,explanation,actual text
0,4,4.0,True,"Despite being empty late, the reviewer enjoyed...",We got here around midnight last Friday... the...
1,5,5.0,True,"The reviewer's friend, being from Louisiana, i...",Brought a friend from Louisiana here. She say...
2,3,4.0,True,The review indicates consistent satisfaction w...,"Every friday, my dad and I eat here. We order ..."
3,1,1.0,True,Dealership misdiagnosed a simple dirt buildup ...,"My husband and I were really, really disappoin..."
4,5,5.0,True,The reviewer expresses strong love for the pla...,Love this place! Was in phoenix 3 weeks for w...
5,4,5.0,True,The reviewer highlights great value due to low...,This hotel is in a good location for getting t...
6,4,5.0,True,"The review is overwhelmingly positive, praisin...",I love that this place has top seafood plates ...
7,4,5.0,True,"The review uses superlative terms ('awesome', ...",Awesome if you like ramen...even awesomer if y...
8,5,5.0,True,The review uses extremely positive language su...,"Great place for a ""home office"" morning. One o..."
9,1,1.0,True,The reviewer explicitly states '1 star for ser...,"1 star for service, but the food is not ok :( ..."


### Prompt Version 2 — Reasoning Prompt
We instruct the model to:
1. Think step-by-step internally
2. Output only final JSON


In [96]:
def prompt_v2(review):
    return f"""
Analyse the review step-by-step internally.
Do NOT reveal your reasoning.
Decide the most likely 1-5 star rating.

Review: "{review}"

Return strictly JSON:
{{
  "predicted_stars": <1-5>,
  "explanation": "<short reasoning>"
}}
"""


Runnint V2 Prompt for data sample 

In [104]:
results_v2 = []

for i, row in tqdm(sample.iterrows(), total=len(sample)):
    full_prompt = prompt_v2(row["text"])
    raw = call_llm(full_prompt)

    try:
        parsed = extract_json(raw)
        predicted = parsed.get("predicted_stars", None)
        valid_json = True
    except:
        predicted = None
        valid_json = False
    
    results_v2.append({
        "actual": row["stars"],
        "pred": predicted,
        "json_valid": valid_json
    })

df_v2 = pd.DataFrame(results_v2)
df_v2




 93%|█████████▎| 14/15 [01:06<00:04,  4.19s/it]

LLM Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}


100%|██████████| 15/15 [01:06<00:00,  4.43s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 21.81163002s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.

,actual,pred,json_valid
0,4,5.0,True
1,5,5.0,True
2,3,5.0,True
3,1,1.0,True
4,5,5.0,True
5,4,5.0,True
6,4,5.0,True
7,4,5.0,True
8,5,5.0,True
9,1,1.0,True


In [99]:
df_v2

""


### Prompt Version 3 — Few-Shot Prompting
Provide examples of classifications.
Improves consistency and JSON formatting.


In [100]:
def prompt_v3(review):
    return f"""
Here are examples:

Example 1:
Review: "Food was cold and tasteless."
Output:
{{"predicted_stars": 1, "explanation": "Bad food quality."}}

Example 2:
Review: "Service was fast and friendly."
Output:
{{"predicted_stars": 5, "explanation": "Excellent service."}}

Now classify this review:

"{review}"

Return strictly valid JSON only.
"""


In [ ]:
# results_v3 = []

# for i, row in tqdm(sample.iterrows(), total=len(sample)):
#     full_prompt = prompt_v3(row["text"])
#     raw = call_llm(full_prompt)

#     try:
#         parsed = extract_json(raw)
#         predicted = parsed.get("predicted_stars", None)
#         explanation = parsed.get("explanation", None)
#         valid_json = True
#     except:
#         predicted = None
#         valid_json = False
    
#     results_v1.append({
#         "actual": row["stars"],
#         "pred": predicted,
#         "json_valid": valid_json,
#         "explanation":explanation,
#         "actual text": row["text"]
#     })

# df_v3 = pd.DataFrame(results_v3)
# df_v3

 80%|████████  | 12/15 [00:31<00:06,  2.04s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 27.780220545s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2

 87%|████████▋ | 13/15 [00:31<00:03,  1.51s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 27.468593289s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2

 93%|█████████▎| 14/15 [00:32<00:01,  1.15s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 27.174755408s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2

100%|██████████| 15/15 [00:32<00:00,  2.17s/it]

LLM Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash\nPlease retry in 26.852803423s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2

""


0

In [ ]:
results_v3 = []

for i, row in tqdm(sample.iterrows(), total=len(sample)):
    full_prompt = prompt_v3(row["text"])
    raw = call_llm(full_prompt)

    try:
        parsed = extract_json(raw)
        predicted = parsed.get("predicted_stars", None)
        explanation = parsed.get("explanation", None)
        valid_json = True
    except:
        predicted = None
        valid_json = False
    
    results_v1.append({
        "actual": row["stars"],
        "pred": predicted,
        "json_valid": valid_json,
        "explanation":explanation,
        "actual text": row["text"]
    })

df_v3 = pd.DataFrame(results_v3)
df_v3


100%|██████████| 15/15 [00:40<00:00,  2.73s/it]


,actual,pred,json_valid
0,4,5,True
1,5,5,True
2,3,4,True
3,1,1,True
4,5,5,True
5,4,5,True
6,4,5,True
7,4,5,True
8,5,5,True
9,1,1,True


### Evaluation

In [85]:
def evaluate(df):
    accuracy = (df["actual"] == df["pred"]).mean()
    json_validity = df["json_valid"].mean()
    missing_preds = df["pred"].isna().sum()
    return accuracy, json_validity, missing_preds


In [86]:
metrics = {
    "Prompt V1": evaluate(df_v1),
    "Prompt V2": evaluate(df_v2),
    "Prompt V3": evaluate(df_v3)
}

metrics


{'Prompt V1': (np.float64(0.4), np.float64(0.7333333333333333), np.int64(4)),
 'Prompt V2': (np.float64(0.4666666666666667), np.float64(0.8), np.int64(3)),
 'Prompt V3': (np.float64(0.5333333333333333), np.float64(1.0), np.int64(0))}

In [87]:
eval_df = pd.DataFrame(metrics, index=["Accuracy", "JSON Validity", "Missing Predictions"])
eval_df.T


,Accuracy,JSON Validity,Missing Predictions
Prompt V1,0.400000,0.733333,4.0
Prompt V2,0.466667,0.800000,3.0
Prompt V3,0.533333,1.000000,0.0


## Discussion of Results

The evaluation of the three prompt versions for Yelp review rating prediction shows a clear improvement in performance as the prompts become more structured and informative.

1. **Prompt V1 (Basic Direct Instruction)**  
   - **Accuracy:** 0.40  
   - **JSON Validity:** 0.73  
   - **Missing Predictions:** 4  
   
   This baseline prompt achieved moderate accuracy and JSON validity. While it instructs the model to return JSON, the lack of guidance for reasoning led to some malformed outputs and inconsistencies. A few predictions were missing due to the model not strictly following the format.

2. **Prompt V2 (Hidden Reasoning / Structured Instruction)**  
   - **Accuracy:** 0.47  
   - **JSON Validity:** 0.80  
   - **Missing Predictions:** 3  
   
   By instructing the model to analyze the review internally before predicting, Prompt V2 improved both accuracy and JSON validity. The structured instructions helped the model reason more carefully, resulting in fewer missing predictions and slightly higher consistency.

3. **Prompt V3 (Few-Shot Examples)**  
   - **Accuracy:** 0.53  
   - **JSON Validity:** 1.00  
   - **Missing Predictions:** 0  
   
   Providing explicit input-output examples allowed the model to follow the expected format precisely, achieving perfect JSON validity and no missing predictions. Accuracy also improved, as the examples guided the model in identifying key indicators for star ratings.


### Key Observations

- **Increasing structure improves reliability:** Adding reasoning instructions (V2) and examples (V3) led to better JSON compliance and fewer missing outputs.  
- **Few-shot examples enhance accuracy:** V3 demonstrates that showing the model explicit patterns for input and output helps it make better predictions.  
- **Trade-off between simplicity and performance:** While V1 is simple, its performance is limited. Adding guidance and examples increases complexity slightly but significantly improves results.

**Conclusion:**  
Structured prompting, particularly with few-shot examples, is the most effective approach for this rating prediction task. Prompt design directly affects accuracy, JSON validity, and the reliability of the model’s predictions. This demonstrates the importance of carefully crafting prompts when using LLMs for structured tasks.
